# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [1]:
import pandas as pd
import numpy as np

tic_tac_toe= pd.read_csv("/tic-tac-toe.csv")
tic_tac_toe.head()

FileNotFoundError: [Errno 2] No such file or directory: '/tic-tac-toe.csv'

In [ ]:
# Convert the first 9 columns (Tic-Tac-Toe grid) to numeric values:
# Ensure all values in the first 9 columns are lowercase and stripped of whitespace
#tic_tac_toe.iloc[:, :9] = tic_tac_toe.iloc[:, :9].apply(lambda x: x.str.strip().str.lower())

# replaxing x with 1, o with -1 and b with 0
tic_tac_toe.replace({'x': 1, 'o': -1, 'b': 0}, inplace=True)
tic_tac_toe['class'] = tic_tac_toe['class'].astype(int)
# Convert the last column (class) to numeric values:
# 'True' (Player X wins) -> 1, 'False' (Player O wins or draw) -> 0
tic_tac_toe.iloc[:, -1] = tic_tac_toe.iloc[:, -1].replace({True: 1, False: 0})

# Display first few rows to check the transformation
print(tic_tac_toe.head())


   TL  TM  TR  ML  MM  MR  BL  BM  BR  class
0   1   1   1   1  -1  -1   1  -1  -1      1
1   1   1   1   1  -1  -1  -1   1  -1      1
2   1   1   1   1  -1  -1  -1  -1   1      1
3   1   1   1   1  -1  -1  -1   0   0      1
4   1   1   1   1  -1  -1   0  -1   0      1


<ipython-input-29-8c6784e3f9f0>:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  tic_tac_toe.replace({'x': 1, 'o': -1, 'b': 0}, inplace=True)


In [ ]:
# Separate inputs (first 9 columns) and output (last column)
X = tic_tac_toe.drop(columns=['class'])  # Inputs: Board positions
y = tic_tac_toe['class']  # Output: Game result (1 = X wins, 0 = O wins or draw)

# Display the shapes of the separated data
print(f"Input shape: {X.shape}")
print(f"Output shape: {y.shape}")

# Display first few rows of inputs and outputs
print("\nFirst 5 Inputs:")
print(X.head())

print("\nFirst 5 Outputs:")
print(y.head())

Input shape: (958, 9)
Output shape: (958,)

First 5 Inputs:
   TL  TM  TR  ML  MM  MR  BL  BM  BR
0   1   1   1   1  -1  -1   1  -1  -1
1   1   1   1   1  -1  -1  -1   1  -1
2   1   1   1   1  -1  -1  -1  -1   1
3   1   1   1   1  -1  -1  -1   0   0
4   1   1   1   1  -1  -1   0  -1   0

First 5 Outputs:
0    1
1    1
2    1
3    1
4    1
Name: class, dtype: int64


In [ ]:
from sklearn.preprocessing import StandardScaler

# Normalize the input features (X) using StandardScaler
scaler = StandardScaler()
X_normalized = scaler.fit_transform(X)

# Convert back to DataFrame for better readability
X_normalized_df = pd.DataFrame(X_normalized, columns=X.columns)

print(X_normalized_df.head())

        TL       TM       TR       ML        MM        MR        BL        BM  \
0  1.03517  1.10683  1.03517  1.10683 -1.241994 -1.223594  1.035170 -1.223594   
1  1.03517  1.10683  1.03517  1.10683 -1.241994 -1.223594 -1.231556  1.106830   
2  1.03517  1.10683  1.03517  1.10683 -1.241994 -1.223594 -1.231556 -1.223594   
3  1.03517  1.10683  1.03517  1.10683 -1.241994 -1.223594 -1.231556 -0.058382   
4  1.03517  1.10683  1.03517  1.10683 -1.241994 -1.223594 -0.098193 -1.223594   

         BR  
0 -1.231556  
1 -1.231556  
2  1.035170  
3 -0.098193  
4 -0.098193  


In [ ]:
for col in tic_tac_toe.columns[:9]:  # Only check first 9 columns
    print(f"Unique values in {col}: {tic_tac_toe[col].unique()}")


Unique values in TL: [ 1 -1  0]
Unique values in TM: [ 1 -1  0]
Unique values in TR: [ 1 -1  0]
Unique values in ML: [ 1 -1  0]
Unique values in MM: [-1  0  1]
Unique values in MR: [-1  0  1]
Unique values in BL: [ 1 -1  0]
Unique values in BM: [-1  1  0]
Unique values in BR: [-1  1  0]


# Nova secção

## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
#
# Split the dataset (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X_normalized, y, test_size=0.2, random_state=42)

model = Sequential()

model.add(Dense(50, activation='relu', input_shape=(9,)))
model.add(Dense(50, activation='relu'))
model.add(Dense(2, activation='softmax'))

# compile model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.summary()



/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                      │ (None, 50)                  │             500 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 50)                  │           2,550 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 2)                   │             102 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,152 (12.31 KB)

 Trainable params: 3,152 (12.31 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# fit the model
model.fit(X_train, y_train, epochs=20, validation_data=(X_test, y_test))



Epoch 1/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.6866 - loss: 0.6127 - val_accuracy: 0.6927 - val_loss: 0.5674
Epoch 2/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7482 - loss: 0.5371 - val_accuracy: 0.7708 - val_loss: 0.5197
Epoch 3/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7687 - loss: 0.5060 - val_accuracy: 0.7708 - val_loss: 0.4933
Epoch 4/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8160 - loss: 0.4510 - val_accuracy: 0.7865 - val_loss: 0.4642
Epoch 5/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8131 - loss: 0.4396 - val_accuracy: 0.8229 - val_loss: 0.4337
Epoch 6/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8455 - loss: 0.4060 - val_accuracy: 0.8698 - val_loss: 0.3950
Epoch 7/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8832 - loss: 0.3489 - val_accuracy: 0.9115 - val_loss: 0.3535
Epoch 8/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9014 - loss: 0.3086 - val_accuracy: 0.9375 - val_los

In [ ]:
# now i want to evaluate the model

test_loss, test_acc = model.evaluate(X_test, y_test)

print(f"Test accurace:", test_acc)
print(f"Test loss:", test_loss)

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9717 - loss: 0.0530 
Test accurace: 0.9791666865348816
Test loss: 0.04439247027039528


In [ ]:
# saving the model for later use. # have to use model.save and import keras.
from tensorflow import keras
model.save("tic-tac-toe.keras")

## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [ ]:
# loading the model

model = keras.models.load_model("tic-tac-toe.keras")

In [ ]:
num_samples = 10

X_test_array = np.array(X_test)
y_test_array = np.array(y_test)

X_sample = X_test_array[np.random.choice(len(X_test_array), num_samples, replace=False)]
y_actual = y_test_array[np.random.choice(len(X_test_array), num_samples, replace=False)]

predictions = model.predict(X_sample)
predicted_classes = np.argmax(predictions, axis=1)

for i in range(num_samples):
    print(f"Actual: {y_actual[i]}, Predicted: {predicted_classes[i]}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
Actual: 0, Predicted: 1
Actual: 0, Predicted: 1
Actual: 1, Predicted: 1
Actual: 1, Predicted: 1
Actual: 1, Predicted: 0
Actual: 1, Predicted: 0
Actual: 1, Predicted: 0
Actual: 1, Predicted: 0
Actual: 1, Predicted: 1
Actual: 0, Predicted: 1


## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [ ]:
model.fit(X_train, y_train, epochs=20, validation_data=(X_test, y_test))

Epoch 1/25
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 1.0000 - loss: 0.0220 - val_accuracy: 0.9844 - val_loss: 0.0373
Epoch 2/25
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.0158 - val_accuracy: 0.9844 - val_loss: 0.0348
Epoch 3/25
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.0144 - val_accuracy: 0.9844 - val_loss: 0.0338
Epoch 4/25
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.0136 - val_accuracy: 0.9844 - val_loss: 0.0321
Epoch 5/25
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.0119 - val_accuracy: 0.9896 - val_loss: 0.0292
Epoch 6/25
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0099 - val_accuracy: 0.9844 - val_loss: 0.0306
Epoch 7/25
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.0091 - val_accuracy: 0.9844 - val_loss: 0.0281
Epoch 8/25
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.0079 - val_accuracy: 0.9844 - val_loss:

**Which approach(es) did you find helpful to improve your model performance?**

In [ ]:
#with double the number of epoch we could get the metrics we were looking for